### Import packages, define paths and load embeddings

In [1]:
from pathlib import Path
from gensim.models.word2vec import Word2Vec
from scipy.stats import spearmanr
import numpy as np
import json
import re
import pandas as pd
from itertools import combinations

In [2]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"
EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"
WORD_SIM353_DIR = PROJECT_ROOT / "data/wordsim353"
TARGET_WORDS_DIR = PROJECT_ROOT / "data/target_words"

In [28]:
# load embeddings
romance_embeddings_2000 = Word2Vec.load(str(EMBEDDINGS_DIR / "romance_2000.w2v"))
sci_fi_embeddings_2000 = Word2Vec.load(str(EMBEDDINGS_DIR / "sci_fi_2000.w2v"))
literary_fiction_embeddings_2000 = Word2Vec.load(str(EMBEDDINGS_DIR / "literary_fiction_2000.w2v"))

In [4]:
# define datapaths
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
lit_fic_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

wordsim_datapath = WORD_SIM353_DIR / "combined.csv"

In [5]:
# Load the wordsim data
wordsim = pd.read_csv(wordsim_datapath)

# load the target words
target_words_romance = json.load(open(TARGET_WORDS_DIR / "target_words_romance.json"))
target_words_sci_fi = json.load(open(TARGET_WORDS_DIR / "target_words_sci_fi.json"))
target_words_lit = json.load(open(TARGET_WORDS_DIR / "target_words_literary_fiction.json"))

In [6]:
# Load the story data
def load_jsonl_data(file_path):
    with open(file_path, 'r') as f:
        stories = [json.loads(line) for line in f]
    return stories

lit_fiction_stories = load_jsonl_data(lit_fic_data_path)
romance_stories = load_jsonl_data(romance_data_path)
sci_fi_stories = load_jsonl_data(sci_fi_data_path)

### Vocabulary size

In [7]:
# calculate amount of unique words for each genre
def unique_words_in_story(text):
    words = re.findall(r"\b\w+\b", text.lower())
    return len(set(words))

lit_unique = [unique_words_in_story(s["story"]) for s in lit_fiction_stories]
romance_unique = [unique_words_in_story(s["story"]) for s in romance_stories]
sci_fi_unique = [unique_words_in_story(s["story"]) for s in sci_fi_stories]


print("Literary fiction mean:", np.mean(lit_unique))
print("Literary fiction sd:", np.std(lit_unique))

print("Romance mean:", np.mean(romance_unique))
print("Romance sd:", np.std(romance_unique))

print("Sci-fi mean:", np.mean(sci_fi_unique))
print("Sci-fi sd:", np.std(sci_fi_unique))

Literary fiction mean: 657.4687931034483
Literary fiction sd: 57.608791162747515
Romance mean: 617.1337878787879
Romance sd: 55.36275258267748
Sci-fi mean: 751.8553571428571
Sci-fi sd: 67.52207686496442


### WordSim353 comparison

In [8]:
# normalize human scores to [0, 1]
wordsim['Human (mean)'] = wordsim['Human (mean)'] / 10.0

In [9]:
def calculate_similarity(embeddings, word1, word2):
    if word1 in embeddings.wv and word2 in embeddings.wv:
        return embeddings.wv.similarity(word1, word2)
    else:
        return None

In [29]:
# compute similarity for all word pairs in wordsim and append to dataframe
similarities_romance = []
for _, row in wordsim.iterrows():
    sim = calculate_similarity(romance_embeddings_2000, row['Word 1'], row['Word 2'])
    similarities_romance.append(sim)

wordsim['CosineSimilarityRomance'] = similarities_romance

similarities_sci_fi = []
for _, row in wordsim.iterrows():
    sim = calculate_similarity(sci_fi_embeddings_2000, row['Word 1'], row['Word 2'])
    similarities_sci_fi.append(sim)

wordsim['CosineSimilaritySciFi'] = similarities_sci_fi

similarities_lit_fic = []
for _, row in wordsim.iterrows():
    sim = calculate_similarity(literary_fiction_embeddings_2000, row['Word 1'], row['Word 2'])
    similarities_lit_fic.append(sim)

wordsim['CosineSimilarityLitFic'] = similarities_lit_fic

In [11]:
# calculate Spearman correlation between human scores and cosine similarities for each genre
def calculate_spearman_correlation(df, human_col, sim_col):
    # drop rows with None values in sim_col
    df = df.dropna(subset=[sim_col])
    if len(df) > 0:
        correlation, _ = spearmanr(df[human_col], df[sim_col])
        return correlation
    else:
        return None
    
correlation_romance = calculate_spearman_correlation(wordsim, 'Human (mean)', 'CosineSimilarityRomance')
correlation_sci_fi = calculate_spearman_correlation(wordsim, 'Human (mean)', 'CosineSimilaritySciFi')
correlation_lit_fic = calculate_spearman_correlation(wordsim, 'Human (mean)', 'CosineSimilarityLitFic')

print(f"Spearman correlation for Romance embeddings: {correlation_romance:.4f}")
print(f"Spearman correlation for Sci-Fi embeddings: {correlation_sci_fi:.4f}")
print(f"Spearman correlation for Literary Fiction embeddings: {correlation_lit_fic:.4f}")

Spearman correlation for Romance embeddings: 0.4630
Spearman correlation for Sci-Fi embeddings: 0.5423
Spearman correlation for Literary Fiction embeddings: 0.5327


In [12]:
wordsim

,Word 1,Word 2,Human (mean),CosineSimilarityRomance,CosineSimilaritySciFi,CosineSimilarityLitFic
0,love,sex,0.677,NaN,NaN,NaN
1,tiger,cat,0.735,0.255498,NaN,0.267534
2,tiger,tiger,1.000,1.000000,NaN,1.000000
3,book,paper,0.746,0.305983,0.406349,0.285911
4,computer,keyboard,0.762,0.578832,NaN,0.562231
...,...,...,...,...,...,...
348,shower,flood,0.603,0.221222,0.298907,0.259031
349,weather,forecast,0.834,0.532484,0.484708,0.630335
350,disaster,area,0.625,0.269986,0.202062,0.323378
351,governor,office,0.634,NaN,0.338953,NaN


### Testing similarity between seeds

In [30]:
seeds = [1, 7, 13, 42, 99, 123, 256, 512, 777, 999, 2000]

romance_models = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"romance_{seed}.w2v"))
    for seed in seeds
}

sci_fi_models = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"sci_fi_{seed}.w2v"))
    for seed in seeds
}

lit_models = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"literary_fiction_{seed}.w2v"))
    for seed in seeds
}

#### Nearest Neighbors Overlap

In [17]:
def neighbor_overlap(model1, model2, word, topn=10):
    n1 = [w for w, _ in model1.wv.most_similar(word, topn=topn)]
    n2 = [w for w, _ in model2.wv.most_similar(word, topn=topn)]
    
    return len(set(n1) & set(n2)) / topn

def average_overlap_across_models(models, words, topn=10):
    pairwise_averages = []

    # iterate over all unique pairs of models
    for m1, m2 in combinations(models, 2):
        scores = [
            neighbor_overlap(m1, m2, w, topn)
            for w in words if w in m1.wv and w in m2.wv
        ]
        if scores:  # avoid empty list
            pairwise_averages.append(np.mean(scores))

    if pairwise_averages:
        return np.mean(pairwise_averages)
    else:
        return np.nan  # in case no words overlap

##### Overlap with gendered words

In [18]:
female_list = ['she', 'daughter', 'hers', 'her', 'mother', 'woman', 'girl', 'herself', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces']
male_list = ['he', 'son', 'his', 'him', 'father', 'man', 'boy', 'himself', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews']

In [31]:
average_overlap_across_models(
    [lit_models[1], lit_models[7], lit_models[13], lit_models[42], lit_models[123], lit_models[256], lit_models[512], lit_models[777], lit_models[999], lit_models[2000]], female_list + male_list)

0.671437908496732

In [32]:
average_overlap_across_models(
    [sci_fi_models[1], sci_fi_models[7], sci_fi_models[13], sci_fi_models[42], sci_fi_models[123], sci_fi_models[256], sci_fi_models[512], sci_fi_models[777], sci_fi_models[999], sci_fi_models[2000]], female_list + male_list)

0.6746825396825398

In [33]:
average_overlap_across_models(
    [romance_models[1], romance_models[7], romance_models[13], romance_models[42], romance_models[123], romance_models[256], romance_models[512], romance_models[777], romance_models[999], romance_models[2000]], female_list + male_list)

0.7063492063492063

##### Overlap with target words

In [24]:
# combine all lists from target words
high_competence_romance = target_words_romance["high_competence"]
low_competence_romance = target_words_romance["low_competence"]
low_warmth_romance = target_words_romance["low_warmth"]
high_warmth_romance = target_words_romance["high_warmth"]

target_words_romance_all = high_competence_romance + low_competence_romance + low_warmth_romance + high_warmth_romance

# combine all lists from target words
high_competence_lit = target_words_lit["high_competence"]
low_competence_lit = target_words_lit["low_competence"]
low_warmth_lit = target_words_lit["low_warmth"]
high_warmth_lit = target_words_lit["high_warmth"]

target_words_lit_all = high_competence_lit + low_competence_lit + low_warmth_lit + high_warmth_lit

# combine all lists from target words
high_competence_sci_fi = target_words_sci_fi["high_competence"]
low_competence_sci_fi = target_words_sci_fi["low_competence"]
low_warmth_sci_fi = target_words_sci_fi["low_warmth"]
high_warmth_sci_fi = target_words_sci_fi["high_warmth"]

target_words_sci_fi_all = high_competence_sci_fi + low_competence_sci_fi + low_warmth_sci_fi + high_warmth_sci_fi

In [34]:
average_overlap_across_models([lit_models[1], lit_models[7], lit_models[13], lit_models[42], lit_models[123], lit_models[256], lit_models[512], lit_models[777], lit_models[999], lit_models[2000]], target_words_lit_all)

0.507007299270073

In [36]:
average_overlap_across_models([sci_fi_models[1], sci_fi_models[7], sci_fi_models[13], sci_fi_models[42], sci_fi_models[123], sci_fi_models[256], sci_fi_models[512], sci_fi_models[777], sci_fi_models[999], sci_fi_models[2000]], target_words_sci_fi_all)

0.5464399092970522

In [37]:
average_overlap_across_models([romance_models[1], romance_models[7], romance_models[13], romance_models[42], romance_models[123], romance_models[256], romance_models[512], romance_models[777], romance_models[999], romance_models[2000]], target_words_romance_all)

0.579502487562189

##### Considering frequency of words

In [38]:
romance_models[42].wv.get_vecattr("he", "count"),romance_models[42].wv.get_vecattr("warm", "count"),romance_models[42].wv.get_vecattr("friendly", "count")

(158316, 6659, 113)